In [ ]:
import pandas #ipython notebook
titanic = pandas.read_csv("titanic_train.csv")
# print(titanic.head(3))
print(titanic.describe())

In [ ]:
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())
print(titanic.describe())

In [ ]:
print(titanic["Sex"].unique())

# Replace all the occurences of male with the number 0.
titanic.loc[titanic["Sex"] == "male", "Sex"] = 0
titanic.loc[titanic["Sex"] == "female", "Sex"] = 1


In [ ]:
print(titanic["Embarked"].unique())
titanic["Embarked"] = titanic["Embarked"].fillna('S')
titanic.loc[titanic["Embarked"] == "S", "Embarked"] = 0
titanic.loc[titanic["Embarked"] == "C", "Embarked"] = 1
titanic.loc[titanic["Embarked"] == "Q", "Embarked"] = 2

In [ ]:
# Import the linear regression class
from sklearn.linear_model import LinearRegression  # ??????
# Sklearn also has a helper that makes it easy to do cross validation
# from sklearn.cross_validation import KFold
from sklearn.model_selection import KFold

# The columns we'll use to predict the target
predictors = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

# Initialize our algorithm class
alg = LinearRegression()
# Generate cross validation folds for the titanic dataset.  It return the row indices corresponding to train and test.
# We set random_state to ensure we get the same splits every time we run this.
print(titanic.shape)
# kf = KFold(titanic.shape[0], n_folds=3, random_state=1)
kf = KFold(n_splits=3, random_state=1)

predictions = []
for train, test in kf.split(titanic):  # titanic.shape[0]  ????array(891)?????????
    # The predictors we're using the train the algorithm.  Note how we only take the rows in the train folds.
    train_predictors = (titanic[predictors].iloc[train,:])
    print(train_predictors.shape)
    # The target we're using to train the algorithm.
    train_target = titanic["Survived"].iloc[train]
    print(train_target.shape)
    # Training the algorithm using the predictors and target.
    # .fit???????????????????,?????????????????
    alg.fit(train_predictors, train_target)
    # We can now make predictions on the test fold
    # print(titanic[predictors].iloc[test,:])
    test_predictions = alg.predict(titanic[predictors].iloc[test,:])
    print(type(test_predictions))
    print(test_predictions.shape)
    predictions.append(test_predictions)
    print(len(predictions))
    print(type(predictions))
print(len(predictions))
# print(predictions)

In [ ]:
print(type(predictions))
import numpy as np


# The predictions are in three separate numpy arrays.  Concatenate them into one.  
# We concatenate them on axis 0, as they only have one axis.
# numpy???numpy.concatenate((a1,a2,...), axis=0)???????????????????a1,a2,...????????
predictions = np.concatenate(predictions, axis=0)  # #axis=1????????????
print(type(predictions))
print(predictions.shape)

# Map predictions to outcomes (only possible outcomes are 1 and 0)
predictions[predictions > 0.5] = 1
predictions[predictions <= 0.5] = 0
# accuracy = sum(predictions[predictions == titanic["Survived"]]) / len(predictions)
print(predictions[predictions == titanic["Survived"]])
accuracy = len(predictions[predictions == titanic["Survived"]]) / len(predictions)
print(accuracy)

In [ ]:
# from sklearn import cross_validation
from sklearn import model_selection
from sklearn.linear_model import LogisticRegression  # ????,??????
# Initialize our algorithm
alg = LogisticRegression(random_state=1, solver = 'liblinear')
# Compute the accuracy score for all the cross validation folds.  (much simpler than what we did before!)
# scores = cross_validation.cross_val_score(alg, titanic[predictors], titanic["Survived"], cv=3)
scores = model_selection.cross_val_score(alg, titanic[predictors], titanic["Survived"], cv=3)
# cross_val_score(model_name, X, y,cv=k)
# ??:?????????????????,??k?????
# K?????(k-fold)?????????k?,??(k-1)???????,??????????,???????????k???,????k?????
# Take the mean of the scores (because we have one for each fold)
# ???????(???????????)
print(scores.mean())

In [ ]:
# titanic_test = pandas.read_csv("test.csv")
titanic_test = pandas.read_csv("titanic_train.csv")
titanic_test["Age"] = titanic_test["Age"].fillna(titanic["Age"].median())
titanic_test["Fare"] = titanic_test["Fare"].fillna(titanic_test["Fare"].median())
titanic_test.loc[titanic_test["Sex"] == "male", "Sex"] = 0 
titanic_test.loc[titanic_test["Sex"] == "female", "Sex"] = 1
titanic_test["Embarked"] = titanic_test["Embarked"].fillna("S")

titanic_test.loc[titanic_test["Embarked"] == "S", "Embarked"] = 0
titanic_test.loc[titanic_test["Embarked"] == "C", "Embarked"] = 1
titanic_test.loc[titanic_test["Embarked"] == "Q", "Embarked"] = 2

In [ ]:
# from sklearn import cross_validation
from sklearn import model_selection
from sklearn.ensemble import RandomForestClassifier

predictors = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

# Initialize our algorithm with the default paramters
# n_estimators is the number of trees we want to make
# min_samples_split is the minimum number of rows we need to make a split
# min_samples_leaf is the minimum number of samples we can have at the place where a tree branch ends (the bottom points of the tree)
alg = RandomForestClassifier(random_state=1, n_estimators=10, min_samples_split=2, min_samples_leaf=1)
# n_estimators=10 ???????????
# ???????????
# min_samples_split=2  ??????????????
# min_samples_leaf=1  ?????????? 

# Compute the accuracy score for all the cross validation folds.  (much simpler than what we did before!)
# kf = cross_validation.KFold(titanic.shape[0], n_folds=3, random_state=1)
kf = model_selection.KFold(n_splits=3, random_state=1)
scores = model_selection.cross_val_score(alg, titanic[predictors], titanic["Survived"], cv=kf.split(titanic))
# cross_val_score(model_name, X, y,cv=k)
# ??:?????????????????,??k?????

# Take the mean of the scores (because we have one for each fold)
print(scores.mean())

In [ ]:
alg = RandomForestClassifier(random_state=1, n_estimators=100, min_samples_split=4, min_samples_leaf=2)
# Compute the accuracy score for all the cross validation folds.  (much simpler than what we did before!)
# kf = cross_validation.KFold(titanic.shape[0], 3, random_state=1)
kf = model_selection.KFold(n_splits=3, random_state=1)
scores = model_selection.cross_val_score(alg, titanic[predictors], titanic["Survived"], cv=kf.split(titanic))

# Take the mean of the scores (because we have one for each fold)
print(scores.mean())

In [ ]:
# Generating a familysize column
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"]

# The .apply method generates a new series
titanic["NameLength"] = titanic["Name"].apply(lambda x: len(x))

In [ ]:
import re

# A function to get the title from a name.
def get_title(name):
    # Use a regular expression to search for a title.  Titles always consist of capital and lowercase letters, and end with a period.
    # ???????????? ????????????????,???????
    title_search = re.search(' ([A-Za-z]+)\.', name)
    # If the title exists, extract and return it.
    # ??????,???????
    if title_search:
        return title_search.group(1)
    return ""

# Get all the titles and print how often each one occurs.
# ??????????????????
titles = titanic["Name"].apply(get_title)
print(pandas.value_counts(titles))

# Map each title to an integer.  Some titles are very rare, and are compressed into the same codes as other titles.
# ????????????? ????????,???????????????
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Dr": 5, "Rev": 6, "Major": 7, "Col": 7, "Mlle": 8, "Mme": 8, "Don": 9, "Lady": 10, "Countess": 10, "Jonkheer": 10, "Sir": 9, "Capt": 7, "Ms": 2}
for k,v in title_mapping.items():
    titles[titles == k] = v

# Verify that we converted everything.
# ?????????????
print(pandas.value_counts(titles))

# Add in the title column.
titanic["Title"] = titles

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif  # ??k?????????
import matplotlib.pyplot as plt
predictors = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "FamilySize", "Title", "NameLength"]

# Perform feature selection
# ??????
# score_func : callable,???????X?y,??????(scores, pvalues)??????????????f_classif,????????????
# k:int or "all", optional, default=10?????topK?????all????????,??????
# f_classif ??/?????ANOVA F???????
selector = SelectKBest(f_classif, k=5)
selector.fit(titanic[predictors], titanic["Survived"])

# Get the raw p-values for each feature, and transform from p-values into scores
# ?????????p?,???p??????
scores = -np.log10(selector.pvalues_)

# Plot the scores.  See how "Pclass", "Sex", "Title", and "Fare" are the best?
# ????? ??? Pclass?,? Sex?,? Title??? Fare??????
plt.bar(range(len(predictors)), scores)
# ?????  bar(x, height, width=0.8, bottom=None, , align='center', data=None, kwargs*)
plt.xticks(range(len(predictors)), predictors, rotation='vertical')  # rotation??lable???????
plt.show()

# Pick only the four best features.
# predictors = ["Pclass", "Sex", "Fare", "Title"]
predictors = ["Pclass", "Sex", "Fare", "Title", "NameLength"]

alg = RandomForestClassifier(random_state=1, n_estimators=50, min_samples_split=8, min_samples_leaf=4)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier  # ensemble??????????
# ?sacikit-learn?,GradientBoostingClassifier?GBDT????, ?GradientBoostingRegressor?GBDT????
# ??,????boosting?????????
# n_estimators: ??????????????,??????????????????n_estimators??,?????,n_estimators??,
# ??????,???????????????100??????????,?????n_estimators????????learning_rate????
# learning_rate: ???????????????,?????
# alpha:??????GradientBoostingRegressor?,?????Huber??"huber"???????quantile??,?????????????0.9,???????,?????????????
# loss: ???GBDT????????????????????????????
import numpy as np

# The algorithms we want to ensemble.
# ????????
# We're using the more linear predictors for the logistic regression, and everything with the gradient boosting classifier.
# ?????????????logistic??,???????????????
algorithms = [
    [GradientBoostingClassifier(random_state=1, n_estimators=25, max_depth=3), ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize", "Title"]],
    [LogisticRegression(random_state=1, solver = 'liblinear'), ["Pclass", "Sex", "Fare", "FamilySize", "Title", "Age", "Embarked"]]
]
# algorithms = [
#     [GradientBoostingClassifier(random_state=1, n_estimators=25, max_depth=3), ["Pclass", "Sex", "Fare", "Title", "NameLength"]],
#     [LogisticRegression(random_state=1, solver = 'liblinear'), ["Pclass", "Sex", "Fare", "Title", "NameLength"]]
# ]

# Initialize the cross validation folds
# kf = KFold(titanic.shape[0], n_folds=3, random_state=1)
kf = KFold(n_splits=3, random_state=1)
# ????,????????????????,????????????????????,??????????????

predictions = []
for train, test in kf.split(titanic):
    train_target = titanic["Survived"].iloc[train]
    full_test_predictions = []
    # Make predictions for each algorithm on each fold
    # ??????????????
    for alg, predictors in algorithms:
        # Fit the algorithm on the training data.
        # ?????????
        alg.fit(titanic[predictors].iloc[train,:], train_target)
        # Select and predict on the test fold.  
        # ?????????
        # The .astype(float) is necessary to convert the dataframe to all floats and avoid an sklearn error.
        # .astype(float)??Dataframe?????float???sklearn??????
        print(alg.predict_proba(titanic[predictors].iloc[test,:]).shape)
        # predict_proba?????? n ? k ????, ? i ? ? j ?????????? ? i ?????????????,??????????1
        test_predictions = alg.predict_proba(titanic[predictors].iloc[test,:].astype(float))[:,1]
        print(test_predictions.shape)
        full_test_predictions.append(test_predictions)
    # Use a simple ensembling scheme -- just average the predictions to get the final classification.
    # ?????????-?????????????????
    # test_predictions = (full_test_predictions[0] + full_test_predictions[1]) / 2
    # test_predictions = (full_test_predictions[0] + full_test_predictions[1]) / 2
    test_predictions = (full_test_predictions[0] * 4 + full_test_predictions[1]) / 5
    print(test_predictions.shape)
    # Any value over .5 is assumed to be a 1 prediction, and below .5 is a 0 prediction.
    # ??.5????????1??,??.5?????0??
    test_predictions[test_predictions <= .5] = 0
    test_predictions[test_predictions > .5] = 1
    print(test_predictions.shape)
    predictions.append(test_predictions)

# Put all the predictions together into one array.
# ?????????????
predictions = np.concatenate(predictions, axis=0)
print(predictions.shape)

# Compute accuracy by comparing to the training data.
# ?????????????????
# accuracy = sum(predictions[predictions == titanic["Survived"]]) / len(predictions)
print(predictions[predictions == titanic["Survived"]])
accuracy = len(predictions[predictions == titanic["Survived"]]) / len(predictions)
print(accuracy)


In [ ]:
titles = titanic_test["Name"].apply(get_title)
# We're adding the Dona title to the mapping, because it's in the test set, but not the training set
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Dr": 5, "Rev": 6, "Major": 7, "Col": 7, "Mlle": 8, "Mme": 8, "Don": 9, "Lady": 10, "Countess": 10, "Jonkheer": 10, "Sir": 9, "Capt": 7, "Ms": 2, "Dona": 10}
for k,v in title_mapping.items():
    titles[titles == k] = v
titanic_test["Title"] = titles
# Check the counts of each unique title.
print(pandas.value_counts(titanic_test["Title"]))

# Now, we add the family size column.
titanic_test["FamilySize"] = titanic_test["SibSp"] + titanic_test["Parch"]
titanic_test["NameLength"] = titanic["Name"].apply(lambda x: len(x))



In [ ]:
# predictors = ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize", "Title"]
# predictors = ["Pclass", "Sex", "Fare", "Title", "NameLength"]
# predictors = ["Pclass", "Sex", "Fare", "Title"]
predictors = ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize", "Title", "NameLength"]

# Boosting ????????(bias)??(variance)??(trade-off)
# Boosting ???????????????,???????????????????????????t??????,????t-1????????
# ?????Gradient Tree Boosting?GBDT-- GradientBoostingClassifier
algorithms = [
    [GradientBoostingClassifier(random_state=1, n_estimators=25, max_depth=3), predictors],
    [LogisticRegression(random_state=1, solver = 'liblinear'), predictors]
]

full_predictions = []
for alg, predictors in algorithms:
    # Fit the algorithm using the full training data.
    # ?????????????
    alg.fit(titanic[predictors], titanic["Survived"])
    # Predict using the test dataset.  We have to convert all the columns to floats to avoid an error.
    # ???????????? ?????????????????????
    print(alg.predict_proba(titanic_test[predictors]).shape)
    predictions = alg.predict_proba(titanic_test[predictors].astype(float))[:,1]
    print(predictions.shape)
    full_predictions.append(predictions)
    print(type(full_predictions))
    print(len(full_predictions))

# The gradient boosting classifier generates better predictions, so we weight it higher.
# ??????????????,??????????
predictions = (full_predictions[0] * 3 + full_predictions[1]) / 4
print(type(predictions))  # ???list???list????????numpy.ndarray????
print(predictions.shape)
# print(predictions[predictions])
## ????????????(????)??
# print(predictions[predictions <= .5])
predictions[predictions <= .5] = 0  # ???????????????,????????
predictions[predictions > .5] = 1
# predictions[test_predictions > .5] = 1
# ???????0?????????; ???891,?????????297

print(predictions == titanic["Survived"])  # ????????????????
print(predictions[predictions == titanic["Survived"]])  # ????????,????????????????Ture??predictions??????
print(len(predictions[predictions == titanic["Survived"]]))
print(predictions[predictions == titanic["Survived"]].shape)
print(sum(predictions[predictions == titanic["Survived"]]))  # ?????????????,?sum??????
print(type(predictions[predictions == titanic["Survived"]]))
# accuracy = sum(predictions[predictions == titanic["Survived"]]) / len(predictions)
# sum????????,??????????????,??0???????????,???????????????????,???????
accuracy = len(predictions[predictions == titanic["Survived"]]) / len(predictions)
print(accuracy)